# 🔬 Gaussian Mixture Models (GMM) & Expectation Maximization (EM)
### End-to-End ML Notebook — From Scratch to Production

---

## Cell 1 — Dataset: Credit Card Customer Segmentation

**Dataset:** [Credit Card Dataset for Clustering — Kaggle](https://www.kaggle.com/datasets/arjunbhasin2013/ccdata)  
**Rows:** ~8,950 customers | **Features:** 18 credit card usage attributes  
**Goal:** Discover latent customer segments using soft probabilistic clustering (GMM)

### Why GMM over K-Means for this dataset?
- Customer behaviour is **naturally overlapping** — a "moderate spender" is simultaneously somewhat a "low spender" and somewhat a "high spender"
- GMM assigns **probabilities** to each customer for each segment, not hard binary labels
- GMM can model **elliptical clusters** of different orientations and sizes

### Dataset Features (key ones)
| Feature | Description |
|---------|-------------|
| `BALANCE` | Account balance |
| `PURCHASES` | Total purchases amount |
| `CREDIT_LIMIT` | Credit limit on card |
| `PAYMENTS` | Total payments made |
| `TENURE` | Number of months with the bank |


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── Load Dataset ──────────────────────────────────────────────────────────────
# Download from: https://www.kaggle.com/datasets/arjunbhasin2013/ccdata
# File name: CC GENERAL.csv  →  place in same folder as this notebook

try:
    df = pd.read_csv('CC GENERAL.csv')
    print("✅ Dataset loaded from disk.")
except FileNotFoundError:
    # Synthetic fallback with same schema for demo purposes
    np.random.seed(42)
    n = 8950
    df = pd.DataFrame({
        'CUST_ID': [f'C{i:05d}' for i in range(1, n+1)],
        'BALANCE': np.abs(np.random.normal(1500, 2000, n)),
        'BALANCE_FREQUENCY': np.random.uniform(0, 1, n),
        'PURCHASES': np.abs(np.random.exponential(1000, n)),
        'ONEOFF_PURCHASES': np.abs(np.random.exponential(600, n)),
        'INSTALLMENTS_PURCHASES': np.abs(np.random.exponential(400, n)),
        'CASH_ADVANCE': np.abs(np.random.exponential(900, n)),
        'PURCHASES_FREQUENCY': np.random.uniform(0, 1, n),
        'ONEOFF_PURCHASES_FREQUENCY': np.random.uniform(0, 1, n),
        'PURCHASES_INSTALLMENTS_FREQUENCY': np.random.uniform(0, 1, n),
        'CASH_ADVANCE_FREQUENCY': np.random.uniform(0, 1, n),
        'CASH_ADVANCE_TRX': np.random.randint(0, 30, n),
        'PURCHASES_TRX': np.random.randint(0, 100, n),
        'CREDIT_LIMIT': np.abs(np.random.normal(4500, 3500, n)),
        'PAYMENTS': np.abs(np.random.exponential(1700, n)),
        'MINIMUM_PAYMENTS': np.abs(np.random.exponential(850, n)),
        'PRC_FULL_PAYMENT': np.random.uniform(0, 1, n),
        'TENURE': np.random.randint(6, 13, n),
    })
    print("⚠️  CC GENERAL.csv not found — using synthetic data with the same schema.")
    print("   Download from: https://www.kaggle.com/datasets/arjunbhasin2013/ccdata\n")

print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")


In [ ]:
# ── Basic Exploration ─────────────────────────────────────────────────────────
print("── First 5 rows ──")
display(df.head())

print("\n── Statistical Summary ──")
display(df.describe().round(2))

print("\n── Missing Values ──")
missing = df.isnull().sum()
print(missing[missing > 0])
print(f"Total missing: {df.isnull().sum().sum()}")


---
## Cell 2 — Theory Recap

### What Is a Gaussian Mixture Model?

A GMM models the data distribution as a **weighted sum of K Gaussian components**:

$$p(x) = \sum_{k=1}^{K} \pi_k \, \mathcal{N}(x \mid \mu_k, \Sigma_k)$$

### The Gaussian (Normal) Distribution

$$\mathcal{N}(x \mid \mu, \Sigma) = \frac{1}{(2\pi)^{d/2}|\Sigma|^{1/2}} \exp\!\left(-\frac{1}{2}(x-\mu)^T \Sigma^{-1} (x-\mu)\right)$$

| Symbol | Meaning |
|--------|---------|
| `πₖ` | **Mixture weight** — how prevalent cluster `k` is; all sum to 1 |
| `μₖ` | **Mean vector** — center of Gaussian `k` |
| `Σₖ` | **Covariance matrix** — shape and orientation of Gaussian `k` |
| `K` | Number of mixture components (your hyperparameter) |

### The EM Algorithm — Two Alternating Steps

**E-Step (Expectation):** Compute the *responsibility* `rᵢₖ` — the posterior probability that component `k` generated data point `xᵢ`:

$$r_{ik} = \frac{\pi_k \, \mathcal{N}(x_i \mid \mu_k, \Sigma_k)}{\sum_{j=1}^{K} \pi_j \, \mathcal{N}(x_i \mid \mu_j, \Sigma_j)}$$

**M-Step (Maximization):** Use responsibilities to update all parameters:

$$\mu_k^{new} = \frac{\sum_i r_{ik}\, x_i}{\sum_i r_{ik}}, \quad
  \Sigma_k^{new} = \frac{\sum_i r_{ik}(x_i-\mu_k)(x_i-\mu_k)^T}{\sum_i r_{ik}}, \quad
  \pi_k^{new} = \frac{\sum_i r_{ik}}{N}$$

### Key Insight: Soft vs Hard Assignment
- **K-Means:** Customer belongs to ONE cluster — hard, binary
- **GMM:** Customer belongs to ALL clusters with PROBABILITIES — soft, nuanced

> *Analogy: A music streaming user is 70% "indie fan", 25% "pop fan", 5% "jazz fan" — GMM captures this; K-Means forces a single label.*


---
## Cell 3 — From-Scratch GMM Implementation (NumPy Only)

We implement the full EM algorithm from scratch so every equation from Cell 2 maps directly to code.

**Implementation notes:**
- Log-space computations to avoid numerical underflow (tiny Gaussian probabilities × tiny probabilities = NaN)
- Covariance regularization (`reg_covar`) to prevent singular matrices when a Gaussian collapses onto one point
- Convergence tracked via log-likelihood improvement


In [ ]:
import numpy as np

class GMMScratch:
    """
    Gaussian Mixture Model with EM, implemented in NumPy only.
    Matches the sklearn GaussianMixture API: fit(), predict(), predict_proba().
    """

    def __init__(self, n_components=3, max_iter=200, tol=1e-4,
                 reg_covar=1e-6, random_state=42):
        self.K           = n_components
        self.max_iter    = max_iter
        self.tol         = tol
        self.reg_covar   = reg_covar   # Ridge-style regularisation on covariances
        self.random_state = random_state
        # Learned parameters
        self.weights_    = None   # π_k  shape (K,)
        self.means_      = None   # μ_k  shape (K, d)
        self.covariances_= None   # Σ_k  shape (K, d, d)
        self.converged_  = False
        self.n_iter_     = 0
        self.lower_bound_= -np.inf

    # ── Gaussian PDF (log-space for numerical stability) ──────────────────────
    def _log_gaussian(self, X, mean, cov):
        """
        Log probability of each row of X under N(mean, cov).
        Returns shape (n,).
        """
        d   = X.shape[1]
        cov_reg = cov + self.reg_covar * np.eye(d)
        try:
            L   = np.linalg.cholesky(cov_reg)
        except np.linalg.LinAlgError:
            # Fallback: heavier regularisation
            L   = np.linalg.cholesky(cov_reg + 1e-3 * np.eye(d))
        diff    = X - mean                                        # (n, d)
        sol     = np.linalg.solve(L, diff.T)                     # (d, n)
        maha    = (sol ** 2).sum(axis=0)                         # (n,)  Mahalanobis²
        log_det = 2 * np.sum(np.log(np.diag(L)))
        return -0.5 * (d * np.log(2 * np.pi) + log_det + maha)  # (n,)

    # ── Initialisation ────────────────────────────────────────────────────────
    def _initialise(self, X):
        rng = np.random.RandomState(self.random_state)
        n, d = X.shape
        idx  = rng.choice(n, size=self.K, replace=False)
        self.means_       = X[idx].copy()
        self.covariances_ = np.stack([np.cov(X.T) + self.reg_covar * np.eye(d)
                                       for _ in range(self.K)])
        self.weights_     = np.ones(self.K) / self.K

    # ── E-Step ────────────────────────────────────────────────────────────────
    def _e_step(self, X):
        """
        Compute responsibility matrix R (n × K).
        r_ik = π_k * N(x_i | μ_k, Σ_k) / Σ_j π_j * N(x_i | μ_j, Σ_j)
        """
        n = X.shape[0]
        log_resp = np.zeros((n, self.K))   # log π_k + log N(x_i|μ_k, Σ_k)

        for k in range(self.K):
            log_resp[:, k] = np.log(self.weights_[k] + 1e-300) +                              self._log_gaussian(X, self.means_[k], self.covariances_[k])

        # Log-sum-exp normalisation → log responsibilities
        log_resp_max = log_resp.max(axis=1, keepdims=True)
        log_norm     = log_resp_max + np.log(
                           np.exp(log_resp - log_resp_max).sum(axis=1, keepdims=True))
        log_resp    -= log_norm

        # Log-likelihood: mean of log normalisation constants
        log_likelihood = log_norm.mean()
        return np.exp(log_resp), log_likelihood   # (n, K), scalar

    # ── M-Step ────────────────────────────────────────────────────────────────
    def _m_step(self, X, R):
        """
        Update π, μ, Σ using responsibilities R (n × K).
        """
        n, d  = X.shape
        Nk    = R.sum(axis=0)                        # effective count per component (K,)

        self.weights_ = Nk / n

        for k in range(self.K):
            # Weighted mean
            self.means_[k] = (R[:, k:k+1] * X).sum(axis=0) / Nk[k]
            # Weighted covariance
            diff           = X - self.means_[k]               # (n, d)
            self.covariances_[k] = (R[:, k] * diff.T) @ diff / Nk[k]
            # Regularise to prevent singularity
            self.covariances_[k] += self.reg_covar * np.eye(d)

    # ── Fit ───────────────────────────────────────────────────────────────────
    def fit(self, X):
        self._initialise(X)
        prev_ll = -np.inf

        for i in range(self.max_iter):
            R, log_ll    = self._e_step(X)
            self._m_step(X, R)
            self.n_iter_ = i + 1

            if abs(log_ll - prev_ll) < self.tol:
                self.converged_ = True
                break
            prev_ll = log_ll

        self.lower_bound_ = prev_ll
        return self

    def predict_proba(self, X):
        R, _ = self._e_step(X)
        return R                          # (n, K) soft responsibilities

    def predict(self, X):
        return self.predict_proba(X).argmax(axis=1)

    def score(self, X):
        """Mean log-likelihood (higher is better)."""
        _, ll = self._e_step(X)
        return ll


print("GMMScratch class defined ✓")
print("Methods: fit(), predict(), predict_proba(), score()")
print("Implements: log-space Gaussian PDF, E-Step, M-Step, covariance regularisation")


---
## Cell 4 — Scikit-Learn GaussianMixture

`sklearn.mixture.GaussianMixture` adds:
- Multiple restarts (`n_init`) to escape local optima
- Four covariance types: `'full'`, `'tied'`, `'diag'`, `'spherical'`
- BIC / AIC for model selection

| `covariance_type` | Shape allowed | Use case |
|------------------|--------------|----------|
| `'full'` | Any ellipse, any orientation | Most flexible — use when data allows |
| `'tied'` | Same shape for all components | Faster, fewer parameters |
| `'diag'` | Axis-aligned ellipses | Good balance of speed/flexibility |
| `'spherical'` | Circles only | Equivalent to soft K-Means |


In [ ]:
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# ── Feature selection and preprocessing ───────────────────────────────────────
FEATURES = ['BALANCE', 'PURCHASES', 'CREDIT_LIMIT', 'PAYMENTS',
            'PURCHASES_FREQUENCY', 'TENURE']

X_raw = df[FEATURES].copy()

# Fill missing values (MINIMUM_PAYMENTS and CREDIT_LIMIT can have NaNs)
X_raw = X_raw.fillna(X_raw.median())

# Scale features
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# PCA to 2D for visualisation and From-Scratch GMM (avoids high-d covariance issues)
pca       = PCA(n_components=2, random_state=42)
X_2d      = pca.fit_transform(X_scaled)

print(f"Feature matrix     : {X_scaled.shape}")
print(f"PCA 2D matrix      : {X_2d.shape}")
print(f"Variance explained : {pca.explained_variance_ratio_.round(3)} "
      f"(total={pca.explained_variance_ratio_.sum():.1%})")


In [ ]:
# ── Fit sklearn GaussianMixture ───────────────────────────────────────────────
N_COMPONENTS = 4   # Justified by BIC in Cell 8

sk_gmm = GaussianMixture(
    n_components=N_COMPONENTS,
    covariance_type='full',   # Each cluster gets its own covariance matrix
    n_init=5,                 # 5 restarts — keep the best log-likelihood
    max_iter=300,
    reg_covar=1e-6,           # Regularisation to prevent singular covariances
    random_state=42
)
sk_gmm.fit(X_2d)

print("── sklearn GaussianMixture Results ──")
print(f"  Converged      : {sk_gmm.converged_}")
print(f"  Iterations     : {sk_gmm.n_iter_}")
print(f"  Log-likelihood : {sk_gmm.lower_bound_:.4f}")
print(f"  BIC            : {sk_gmm.bic(X_2d):.2f}")
print(f"  AIC            : {sk_gmm.aic(X_2d):.2f}")
print(f"\nMixture weights (πₖ): {sk_gmm.weights_.round(4)}")
print(f"Cluster sizes (hard): {dict(zip(*np.unique(sk_gmm.predict(X_2d), return_counts=True)))}")


---
## Cell 5 — Data Preprocessing Deep-Dive

### Why Preprocessing Matters for GMM

GMMs fit Gaussian components using Euclidean-space covariance. Two preprocessing issues can silently destroy results:

1. **Unscaled features** — `CREDIT_LIMIT` (range 0–30,000) will dominate `TENURE` (range 6–12); the Gaussians will stretch entirely along the credit limit axis
2. **Missing values** — EM cannot handle NaNs; they must be imputed before fitting
3. **Heavy-tailed distributions** — Credit card data is right-skewed; log-transform normalises it closer to Gaussian, which the model assumes

### Preprocessing Pipeline


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ── Step 1: Missing value report ──────────────────────────────────────────────
print("── Step 1: Missing Values ──")
missing_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
print(missing_pct[missing_pct > 0].to_string())

# ── Step 2: Impute with median (robust to outliers) ──────────────────────────
df_clean = df[FEATURES].fillna(df[FEATURES].median())
print(f"\n── Step 2: After imputation — missing: {df_clean.isnull().sum().sum()}")

# ── Step 3: Log-transform skewed financial features ──────────────────────────
LOG_FEATURES = ['BALANCE', 'PURCHASES', 'CREDIT_LIMIT', 'PAYMENTS']
for col in LOG_FEATURES:
    df_clean[col] = np.log1p(df_clean[col])   # log(1+x) handles zeros safely
print(f"\n── Step 3: Log-transformed: {LOG_FEATURES}")

# ── Step 4: Scale all features ────────────────────────────────────────────────
scaler_clean   = StandardScaler()
X_clean_scaled = scaler_clean.fit_transform(df_clean)
print(f"\n── Step 4: Scaled matrix shape: {X_clean_scaled.shape}")
print(f"   Mean per feature (should ≈ 0): {X_clean_scaled.mean(axis=0).round(3)}")
print(f"   Std  per feature (should ≈ 1): {X_clean_scaled.std(axis=0).round(3)}")

# ── Step 5: PCA 2D for downstream From-Scratch GMM ───────────────────────────
pca_clean     = PCA(n_components=2, random_state=42)
X_clean_2d    = pca_clean.fit_transform(X_clean_scaled)
print(f"\n── Step 5: PCA 2D — variance explained: "
      f"{pca_clean.explained_variance_ratio_.sum():.1%}")

# Reassign working variable for subsequent cells
X_2d = X_clean_2d
print(f"\nPreprocessing pipeline complete ✓  →  X_2d shape: {X_2d.shape}")


---
## Cell 6 — Evaluation: Log-Likelihood, BIC, AIC, Silhouette

### Metrics for Unsupervised Clustering

| Metric | What it measures | Better when |
|--------|-----------------|-------------|
| **Log-Likelihood** | How well the fitted distribution explains the data | Higher |
| **BIC** (Bayesian Info Criterion) | Log-likelihood penalised for model complexity | Lower |
| **AIC** (Akaike Info Criterion) | Like BIC but lighter penalty — favours more components | Lower |
| **Silhouette Score** | Geometric cluster separation (-1 to +1) | Higher |

### Scratch vs Sklearn Comparison


In [ ]:
import time
from sklearn.metrics import silhouette_score
from sklearn.mixture import GaussianMixture

K = 4  # working component count

# ── Retrain sklearn GMM on clean X_2d ─────────────────────────────────────────
sk_gmm = GaussianMixture(n_components=K, covariance_type='full',
                          n_init=5, max_iter=300, reg_covar=1e-6, random_state=42)
sk_gmm.fit(X_2d)

# ── Train From-Scratch GMM ────────────────────────────────────────────────────
print("── From-Scratch GMM ──")
t0          = time.time()
scratch_gmm = GMMScratch(n_components=K, max_iter=200, tol=1e-4,
                          reg_covar=1e-6, random_state=42)
scratch_gmm.fit(X_2d)
t_scratch   = time.time() - t0

scratch_labels = scratch_gmm.predict(X_2d)
scratch_ll     = scratch_gmm.score(X_2d)
scratch_sil    = silhouette_score(X_2d, scratch_labels)

print(f"  Converged       : {scratch_gmm.converged_}")
print(f"  Iterations      : {scratch_gmm.n_iter_}")
print(f"  Log-likelihood  : {scratch_ll:.4f}")
print(f"  Silhouette Score: {scratch_sil:.4f}")
print(f"  Wall-clock time : {t_scratch*1000:.1f} ms")

print()

# ── sklearn GMM ───────────────────────────────────────────────────────────────
print("── Scikit-Learn GMM ──")
t0       = time.time()
sk_gmm.fit(X_2d)
t_sk     = time.time() - t0

sk_labels = sk_gmm.predict(X_2d)
sk_ll     = sk_gmm.score(X_2d)
sk_sil    = silhouette_score(X_2d, sk_labels)

print(f"  Converged       : {sk_gmm.converged_}")
print(f"  Iterations      : {sk_gmm.n_iter_}")
print(f"  Log-likelihood  : {sk_ll:.4f}")
print(f"  BIC             : {sk_gmm.bic(X_2d):.2f}")
print(f"  AIC             : {sk_gmm.aic(X_2d):.2f}")
print(f"  Silhouette Score: {sk_sil:.4f}")
print(f"  Wall-clock time : {t_sk*1000:.1f} ms")

# ── K-Means for comparison ────────────────────────────────────────────────────
from sklearn.cluster import KMeans
km           = KMeans(n_clusters=K, init='k-means++', n_init=10, random_state=42)
km.fit(X_2d)
km_sil       = silhouette_score(X_2d, km.labels_)

print()
print("── Summary Table ──")
print(f"{'Metric':<25} {'Scratch GMM':>14} {'Sklearn GMM':>14} {'K-Means':>14}")
print("─" * 70)
print(f"{'Log-Likelihood':<25} {scratch_ll:>14.4f} {sk_ll:>14.4f} {'N/A':>14}")
print(f"{'Silhouette Score':<25} {scratch_sil:>14.4f} {sk_sil:>14.4f} {km_sil:>14.4f}")
print(f"{'Soft assignments?':<25} {'Yes':>14} {'Yes':>14} {'No':>14}")
print(f"{'Wall-clock (ms)':<25} {t_scratch*1000:>14.1f} {t_sk*1000:>14.1f} {'—':>14}")


---
## Cell 7 — Visualisation

### Plot 1 — Soft Probability Heatmap
Each customer is coloured by their **maximum cluster probability** from GMM.
Customers near 1.0 are clearly in one cluster; customers near 0.5 are on the boundary — GMM captures this ambiguity explicitly.

### Plot 2 — GMM Gaussian Ellipses vs K-Means Hard Boundaries
GMM contours show the actual Gaussian shape (ellipse orientation comes from the covariance matrix).
K-Means Voronoi boundaries are straight lines — a much stronger assumption.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Ellipse
import numpy as np

COLORS  = ['#E63946', '#457B9D', '#2A9D8F', '#E9C46A']
PALETTE = [COLORS[i] for i in range(K)]

# ── Soft probabilities ────────────────────────────────────────────────────────
probs        = sk_gmm.predict_proba(X_2d)          # (n, K)
max_proba    = probs.max(axis=1)                    # confidence in best cluster
hard_labels  = probs.argmax(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(17, 6))
fig.suptitle('GMM Soft Clustering — Credit Card Customer Segmentation',
             fontsize=14, fontweight='bold', y=1.01)

# ── Left: Soft probability scatter ───────────────────────────────────────────
ax1 = axes[0]
sc  = ax1.scatter(X_2d[:, 0], X_2d[:, 1],
                   c=max_proba, cmap='RdYlGn', alpha=0.6, s=12, vmin=0.25, vmax=1.0)
plt.colorbar(sc, ax=ax1, label='Max cluster probability (confidence)')
ax1.set_title('Soft Assignment Confidence\n(green=certain, red=boundary)', fontsize=11, fontweight='bold')
ax1.set_xlabel('PCA Component 1', fontsize=10)
ax1.set_ylabel('PCA Component 2', fontsize=10)
ax1.grid(True, alpha=0.25)

# Overlay cluster centroids
for k in range(K):
    cx, cy = sk_gmm.means_[k]
    ax1.scatter(cx, cy, c='black', marker='*', s=250, zorder=5)
    ax1.annotate(f'μ{k}', (cx, cy), textcoords='offset points', xytext=(6, 4), fontsize=9)

# ── Right: GMM Gaussian ellipses vs K-Means ──────────────────────────────────
ax2 = axes[1]

# K-Means background colours
from sklearn.cluster import KMeans
km = KMeans(n_clusters=K, init='k-means++', n_init=10, random_state=42)
km.fit(X_2d)

x_min, x_max = X_2d[:, 0].min() - 0.5, X_2d[:, 0].max() + 0.5
y_min, y_max = X_2d[:, 1].min() - 0.5, X_2d[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                      np.linspace(y_min, y_max, 300))
Z = km.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
ax2.contourf(xx, yy, Z, alpha=0.12, cmap='tab10', levels=K-1)

# Scatter points
for k in range(K):
    mask = hard_labels == k
    ax2.scatter(X_2d[mask, 0], X_2d[mask, 1],
                c=PALETTE[k], alpha=0.55, s=12, label=f'GMM Cluster {k} (n={mask.sum()})')

# GMM Gaussian ellipses (1σ and 2σ contours)
def plot_gaussian_ellipse(ax, mean, cov, color, n_std=2.0, alpha=0.7):
    vals, vecs = np.linalg.eigh(cov)
    order      = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    angle      = np.degrees(np.arctan2(vecs[1, 0], vecs[0, 0]))
    for std_mult in [1.0, 2.0]:
        width, height = 2 * std_mult * np.sqrt(vals)
        ell = Ellipse(xy=mean, width=width, height=height, angle=angle,
                      edgecolor=color, facecolor='none', linewidth=2.0,
                      linestyle='--' if std_mult == 2 else '-', alpha=alpha)
        ax.add_patch(ell)

for k in range(K):
    plot_gaussian_ellipse(ax2, sk_gmm.means_[k], sk_gmm.covariances_[k], PALETTE[k])
    ax2.scatter(*sk_gmm.means_[k], c='black', marker='*', s=250, zorder=5)

ax2.set_xlim(x_min, x_max); ax2.set_ylim(y_min, y_max)
ax2.set_title('GMM Gaussian Ellipses (1σ/2σ) vs\nK-Means Hard Boundaries (shaded)', fontsize=11, fontweight='bold')
ax2.set_xlabel('PCA Component 1', fontsize=10)
ax2.set_ylabel('PCA Component 2', fontsize=10)
ax2.legend(fontsize=8, loc='upper right')
ax2.grid(True, alpha=0.25)

plt.tight_layout()
plt.savefig('gmm_visualisation.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved → gmm_visualisation.png")


In [ ]:
# ── Visualisation 3: Per-cluster soft probability distributions ──────────────
fig, axes = plt.subplots(1, K, figsize=(16, 4))
fig.suptitle('Soft Responsibility Distribution per GMM Component\n'
             '(How "certain" the model is about each cluster membership)',
             fontsize=12, fontweight='bold')

for k in range(K):
    ax   = axes[k]
    resp = probs[:, k]
    ax.hist(resp, bins=40, color=PALETTE[k], alpha=0.8, edgecolor='black', linewidth=0.3)
    ax.axvline(resp.mean(), color='black', linestyle='--', linewidth=1.5,
               label=f'Mean = {resp.mean():.2f}')
    ax.set_title(f'Component {k}\n(n={int((hard_labels==k).sum())})', fontweight='bold')
    ax.set_xlabel('P(cluster k | xᵢ)', fontsize=9)
    ax.set_ylabel('Count' if k == 0 else '', fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('gmm_responsibilities.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved → gmm_responsibilities.png")
print("\nInterpretation: A spike near 1.0 = this cluster has many highly-confident members.")
print("A spread distribution = many customers are ambiguously between clusters — GMM captures this!")


---
## Cell 8 — Hyperparameter Experiments: Choosing n_components

### Model Selection via BIC and AIC

For GMMs, **BIC** (Bayesian Information Criterion) is the standard way to choose K:

$$\text{BIC} = -2 \ln(\hat{L}) + p \ln(n)$$

where `p` = number of free parameters and `n` = number of data points.

- BIC **penalises complexity more aggressively** than AIC — tends to pick simpler models
- AIC **rewards fit more** — tends to pick more components
- **Best practice:** pick K where both BIC and log-likelihood show diminishing returns

We also compare different `covariance_type` options to see which fits best.


In [ ]:
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score
import numpy as np

K_range = range(2, 10)
results = []

print("Running GMM experiments for K = 2 to 9 ...")
for k in K_range:
    gmm = GaussianMixture(n_components=k, covariance_type='full',
                           n_init=3, max_iter=200, reg_covar=1e-6, random_state=42)
    gmm.fit(X_2d)
    labels = gmm.predict(X_2d)
    sil    = silhouette_score(X_2d, labels) if k > 1 else np.nan
    results.append({
        'K': k,
        'BIC': gmm.bic(X_2d),
        'AIC': gmm.aic(X_2d),
        'Log-Likelihood': gmm.score(X_2d),
        'Silhouette': sil,
        'Converged': gmm.converged_,
    })
    print(f"  K={k} | BIC={gmm.bic(X_2d):8.1f} | AIC={gmm.aic(X_2d):8.1f} | "
          f"LL={gmm.score(X_2d):.4f} | Sil={sil:.4f}")

import pandas as pd
results_df = pd.DataFrame(results)
best_k_bic = results_df.loc[results_df['BIC'].idxmin(), 'K']
best_k_sil = results_df.loc[results_df['Silhouette'].idxmax(), 'K']
print(f"\n  Best K by BIC              : K = {best_k_bic}")
print(f"  Best K by Silhouette Score : K = {best_k_sil}")


In [ ]:
# ── Plot BIC / AIC / Silhouette ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('GMM Hyperparameter Selection: n_components', fontsize=14, fontweight='bold')

K_list = list(K_range)

# BIC
ax1 = axes[0]
ax1.plot(K_list, results_df['BIC'], 'o-', color='#E63946', lw=2.5, ms=8, label='BIC')
ax1.plot(K_list, results_df['AIC'], 's--', color='#457B9D', lw=2.0, ms=7, label='AIC')
ax1.axvline(best_k_bic, color='green', linestyle=':', lw=2, label=f'Best K={best_k_bic} (BIC)')
ax1.set_title('BIC & AIC vs K\n(lower = better)', fontweight='bold')
ax1.set_xlabel('n_components (K)'); ax1.set_ylabel('Score')
ax1.legend(); ax1.grid(True, alpha=0.3)

# Log-Likelihood
ax2 = axes[1]
ax2.plot(K_list, results_df['Log-Likelihood'], 'o-', color='#2A9D8F', lw=2.5, ms=8)
ax2.fill_between(K_list, results_df['Log-Likelihood'], alpha=0.15, color='#2A9D8F')
ax2.set_title('Log-Likelihood vs K\n(higher = better, watch for elbow)', fontweight='bold')
ax2.set_xlabel('n_components (K)'); ax2.set_ylabel('Mean Log-Likelihood')
ax2.grid(True, alpha=0.3)

# Silhouette
ax3 = axes[2]
bar_colors = ['#E9C46A' if k == best_k_sil else '#A8DADC' for k in K_list]
bars = ax3.bar(K_list, results_df['Silhouette'], color=bar_colors, edgecolor='black', alpha=0.85)
ax3.axvline(best_k_sil, color='#E63946', linestyle='--', lw=2, label=f'Best K={best_k_sil}')
ax3.set_title('Silhouette Score vs K\n(higher = better separation)', fontweight='bold')
ax3.set_xlabel('n_components (K)'); ax3.set_ylabel('Silhouette Score')
ax3.legend(); ax3.grid(axis='y', alpha=0.3)
for bar, val in zip(bars, results_df['Silhouette']):
    ax3.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
             f'{val:.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('gmm_hyperparams.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\nConclusion: BIC suggests K={best_k_bic}; Silhouette suggests K={best_k_sil}.")
print("Use domain knowledge to break ties — in customer segmentation, 4-5 is highly actionable.")


---
## Cell 9 — Interview Corner 🎤

The five GMM + EM questions that come up most often in ML engineering interviews.

---

### Q1: What is the fundamental difference between K-Means and GMM?

**Model Answer:**  
K-Means is a **hard assignment** algorithm — each point belongs to exactly one cluster. It implicitly assumes clusters are spherical, equally sized, and equally dense.

GMM is a **soft assignment** probabilistic model — each point has a probability distribution over all clusters. It can model clusters of **any elliptical shape, size, and orientation** via the covariance matrix. K-Means is actually a special case of GMM with fixed, equal, spherical covariances and infinitely sharp responsibilities (converging to 0/1).

---

### Q2: Explain the E-Step and M-Step. What does each one optimize?

**Model Answer:**  
- **E-Step (Expectation):** Given current parameters (μ, Σ, π), compute the **expected cluster responsibilities** `rᵢₖ` via Bayes' theorem. This is the "soft assignment" step — we estimate which component likely generated each point.
- **M-Step (Maximization):** Given fixed responsibilities, **maximize the expected log-likelihood** by analytically updating μ, Σ, and π using weighted means and weighted covariances.

Each full E+M cycle is guaranteed to **monotonically increase the log-likelihood**. The algorithm converges to a local maximum, not necessarily the global one.

---

### Q3: Is EM guaranteed to find the global optimum? How do you mitigate this?

**Model Answer:**  
No — EM converges to a **local maximum** of the log-likelihood. The solution depends on initialization.

Mitigations:
1. **Multiple restarts** (`n_init` in sklearn) — run EM N times from different starting points, keep the best log-likelihood
2. **K-Means warm start** — initialize GMM means from K-Means centroids; this usually finds a good basin quickly
3. **Deterministic seeding** — `random_state` for reproducibility during debugging

---

### Q4: What is a degenerate GMM solution? How do you detect and fix it?

**Model Answer:**  
A degenerate solution occurs when one Gaussian "collapses" onto a single data point — its variance → 0, making its density → ∞, which gives artificially infinite log-likelihood.

**Detect:** Check if any covariance matrix has near-zero determinant; or if log-likelihood diverges to +∞ during training.

**Fix:**
- Add `reg_covar` (regularisation): `Σₖ += λI` at each M-step — this is a ridge penalty on the covariance
- Use `covariance_type='diag'` or `'tied'` — fewer free parameters, less prone to collapse
- Remove outliers before fitting

---

### Q5: How do you choose the number of components K for a GMM? Compare to K-Means.

**Model Answer:**  

| Method | GMM | K-Means |
|--------|-----|---------|
| Information criterion | **BIC / AIC** (penalised log-likelihood) — pick lowest | Not applicable |
| Geometric measure | Silhouette Score | Silhouette Score + Elbow Method |
| Model probability | Bayesian GMM (`BayesianGaussianMixture`) — learns K | Not applicable |
| Domain knowledge | ✅ Always useful | ✅ Always useful |

BIC is preferred over AIC for GMMs in practice because it penalises model complexity more aggressively — AIC tends to overfit by choosing too many components. For customer segmentation, interpretability often caps K at 5–8 regardless of what BIC says.


---
## Cell 10 — Key Takeaways ✅

| # | Takeaway |
|---|----------|
| 1 | **GMM = probabilistic K-Means** — same spirit, but soft assignments and full covariance structure |
| 2 | **Always scale features** — GMM covariance estimation is dominated by high-magnitude features |
| 3 | **Use `reg_covar`** — even a small `1e-6` prevents singular covariance matrices in production |
| 4 | **BIC selects K** — lower BIC = better balance of fit and complexity; prefer BIC over AIC for parsimony |
| 5 | **EM converges to a local max** — use `n_init ≥ 5` and K-Means warm starts to mitigate |
| 6 | **Soft probabilities are the real output** — `predict_proba()` is more useful than `predict()` for overlapping clusters |
| 7 | **`covariance_type='full'`** is the most flexible; `'diag'` is a good default for high-dimensional data |
| 8 | **GMM is a generative model** — you can sample new data from it: `gmm.sample(n_samples=100)` |
| 9 | **Log-space arithmetic is essential** in the from-scratch implementation — naive probability products underflow to 0 |
| 10 | **GMM fails at high dimensionality** without regularisation — full covariance has O(d²) parameters per component |

---

### When to Reach for Alternatives

```
Clusters are highly non-Gaussian?          → DBSCAN / HDBSCAN
Speed + scale matters (millions of rows)?  → K-Means (MiniBatchKMeans)
Full hierarchy of clusters needed?         → Agglomerative Clustering
K is truly unknown?                        → BayesianGaussianMixture (learns K)
Anomaly detection only?                    → Isolation Forest or low-likelihood GMM threshold
```


---
## Cell 11 — References & Further Reading 📚

| Resource | Link |
|----------|------|
| **Dataset** — CC Customer Segmentation (Kaggle) | [kaggle.com](https://www.kaggle.com/datasets/arjunbhasin2013/ccdata) |
| **Original EM Paper** — Dempster, Laird & Rubin (1977) | [JSTOR](https://www.jstor.org/stable/2984875) |
| **Bishop PRML Ch. 9** — EM & Mixture Models | [Microsoft Research PDF](https://www.microsoft.com/en-us/research/uploads/prod/2006/01/Bishop-Pattern-Recognition-and-Machine-Learning-2006.pdf) |
| **sklearn GaussianMixture Docs** | [scikit-learn.org](https://scikit-learn.org/stable/modules/mixture.html) |
| **StatQuest: GMM & EM** | [YouTube](https://www.youtube.com/watch?v=Q2cwkQ521gs) |
| **Kaggle Notebook — GMM for Segmentation** | [kaggle.com](https://www.kaggle.com/code/teunbrand/mixture-models-a-gentle-introduction) |

---

*Notebook authored in the style of MIT 6.867 Machine Learning curriculum.*  
*Series: K-Means → **GMM & EM** → DBSCAN & HDBSCAN → Hierarchical Clustering*
